# В чём идея

Мы хотим повторить успех NARDINI, но для других организмов, и попытаться сопоставить кластеры для разных тварей. Они сами, видимо,сделали то же самое, но пока я вот саму публикацию не видела, да и ручками потрогать тоже хочется.

1. Получим IDRом (совокупность IDR длины >= 30)
2. Получим значения параметров
3. Получим z-скоры, центрировав и нормировав эти значения
4. Кластеризуем данные

Обнаружилась проблема: в протеоме помимо нормальных белков валяется всякая странная фигня, включая "белок" из одного метионина. Наверное, смело можно к чертям выкинуть всё, что короче 30 аминокислот.

In [24]:
# Я как-то тупо её написала, мне не нравится, но оно работает
def pretty_print(seq, file=None):
    l = 60
    i = 0
    for i in range(len(seq) // l):
        print(seq[i * l : (i+1) * l], file=file)
    if len(seq) > i * l and seq[i * l + l:]:
        print(seq[i * l + l:], file=file)

In [25]:
import tqdm

proteome_path = "/mnt/scratch/NPM/genomes/Download/Arabidopsis_thaliana/ncbi_dataset/data/GCF_000001735.4/protein.faa"
out_path = "/home/sasha_suvorova/NPM/IDRome/filtered_proteome.out"

with open(proteome_path) as in_file:
    with open(out_path, 'w') as out:
        name = ''
        seq = ''
        for line in tqdm.tqdm(in_file):
            line = line.rstrip()
            if line.startswith('>'):
                name = line
                if name and len(seq) >= 30:
                    print(name, file=out)
                    pretty_print(seq, file=out)
                    seq = ''
            else:
                seq += line
            

332688it [00:00, 745754.87it/s]


Сделала вот это:

``` bash
(base) sasha_suvorova@hpc2:~/NPM/IDRome$ mobidb_lite filtered_proteome.out mobidblite.out
```

Вроде сработало. Теперь надо распарсить выход

In [19]:
mbdb_path = "/home/sasha_suvorova/NPM/IDRome/mobidblite.out"
prot_path = "/home/sasha_suvorova/NPM/IDRome/filtered_proteome.out"
out_path = "/home/sasha_suvorova/NPM/IDRome/IDRome.faa"

with open(mbdb_path) as coord_file, \
    open(prot_path) as seq_file, \
    open(out_path, 'w') as out:
    name_line = seq_file.readline()[1:]
    seq_gene, descr = name_line.split(maxsplit=1)
    seq = ''
    seq_line = seq_file.readline().rstrip()
    while not seq_line.startswith('>'):
        seq += seq_line
        seq_line = seq_file.readline().rstrip()
    name_line = seq_line[1:]

    count = 0

    for line in coord_file:
        line = line.rstrip()
        idr_gene, start, end, category = line.split(maxsplit=3)
        start = int(start)
        end = int(end)
        if end - start + 1 >= 30:
            count += 1
            while seq_gene != idr_gene:
                seq_gene, descr = name_line.split(maxsplit=1)
                seq = ''
                seq_line = seq_file.readline().rstrip()
                while not seq_line.startswith('>'):
                    seq += seq_line
                    seq_line = seq_file.readline().rstrip()
                name_line = seq_line[1:]

            idr_seq = seq[start - 1: end]
            print(f'>{idr_gene} {start}..{end} {category} {descr}', file=out)
            pretty_print(idr_seq, file=out)

print(count)


28916
